In [ ]:
# import zipfile

# # Path to the zip file you want to extract
# zip_file_path = "Group_16.zip"

# # Path to the directory where you want to extract the contents
# destination_dir = ""

# # Extract the archive while preserving the directory structure
# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     zip_ref.extractall(destination_dir)

# print(f"Successfully unzipped to '{destination_dir}' with directories intact!")

Successfully unzipped to '' with directories intact!


In [ ]:
!pip list

Package                                  Version
---------------------------------------- -------------------
a2a-sdk                                  0.3.26
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 22.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.5
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime       

In [ ]:
import os
import copy
import json
import numpy as np
import random
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.optim import SGD, Adam, Adagrad, RMSprop
from torchvision import datasets, transforms

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")


LEARNING_RATE = 0.001
MOMENTUM = 0.9
epochs = 10000
THRESHOLD = 1e-4

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


ACTIVATION = "tanh"

criterion = nn.CrossEntropyLoss()

ARCHITECTURES = {
    3: {
        "config_1": [128, 64, 32],
        "config_2": [256, 128, 64],
        "config_3": [512, 256, 128]
    }
}

Using GPU: Tesla T4


In [ ]:
data_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(
    'Group_16/train',
    transform=data_transforms
)

val_data = datasets.ImageFolder(
    'Group_16/val',
    transform=data_transforms
)

test_data = datasets.ImageFolder(
    'Group_16/test',
    transform=data_transforms
)

print(train_data.class_to_idx)
print(val_data.class_to_idx)
print(test_data.class_to_idx)

assert train_data.class_to_idx == val_data.class_to_idx
assert train_data.class_to_idx == test_data.class_to_idx

print("Class mapping verified:", train_data.class_to_idx)

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Test samples:", len(test_data))

image, label = train_data[0]

print("Image shape:", image.shape)

{'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
{'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
{'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
Class mapping verified: {'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
Training samples: 11385
Validation samples: 3795
Test samples: 3795
Image shape: torch.Size([1, 28, 28])


# Optimizers and corresponding configs that will be used for each model

In [ ]:
def sgd(model):
  return SGD(model.parameters(),
             lr=LEARNING_RATE)

def sgd_general_delta(model):
  return SGD(model.parameters(),
             lr=LEARNING_RATE,
             momentum=MOMENTUM)

def sgd_nag(model):
  return SGD(model.parameters(),
             lr=LEARNING_RATE,
             momentum=MOMENTUM,
             nesterov=True)

def adagrad(model):
  return Adagrad(model.parameters(),
                 lr=LEARNING_RATE,
                 eps=1e-08
                 )

def rmsprop(model):
  return RMSprop(model.parameters(),
                lr=LEARNING_RATE,
                eps=1e-08,
                alpha=0.99,
                )

def adam(model):
  return Adam(model.parameters(),
              lr=LEARNING_RATE,
              betas=(0.9, 0.999),
              eps=1e-08
              )

optimizers={
    'sgd': sgd,
    'vanillaGD':sgd,
    'sgd_general_delta': sgd_general_delta,
    'sgd_nag': sgd_nag,
    'adagrad': adagrad,
    'rmsprop': rmsprop,
    'adam':adam
}

optimizer_config={
    'sgd':{
        'batch_size':1
    },
    'vanillaGD':{
        'batch_size':len(train_data)
    },
    'sgd_general_delta':{
        'batch_size':1
    },
    'sgd_nag':{
        'batch_size':1
    },
    'adagrad':{
        'batch_size':len(train_data)
    },
    'rmsprop':{
        'batch_size':len(train_data)
    },
    'adam':{
        'batch_size':1
    }
}

In [ ]:
def build_model(hidden_nodes):

    layers = [nn.Flatten()]

    for i in range(len(hidden_nodes)):

        if i == 0:
            layers.append(
                nn.Linear(784, hidden_nodes[i])
            )
        else:
            layers.append(
                nn.Linear(
                    hidden_nodes[i - 1],
                    hidden_nodes[i]
                )
            )

        layers.append(nn.Tanh())

    layers.append(
        nn.Linear(hidden_nodes[-1], 5)
    )

    return nn.Sequential(*layers)

In [ ]:
os.makedirs("initial", exist_ok=True)

for architecture in ARCHITECTURES:

    for config_name, hidden_nodes in ARCHITECTURES[architecture].items():

        output_path = (
            f"initial/model_{architecture}_{config_name}.pth"
        )

        if os.path.exists(output_path):

            print(f"Already exists: {output_path}")

        else:

            model = build_model(hidden_nodes)

            torch.save(
                model.state_dict(),
                output_path
            )

            print(f"Created: {output_path}")

Already exists: initial/model_3_config_1.pth
Already exists: initial/model_3_config_2.pth
Already exists: initial/model_3_config_3.pth


In [ ]:
# best_local_val_accuracy = -1
# best_local_model = None
# best_local_optimizer = None
# best_local_config = None

In [ ]:
def calculate_average_error(model, data):

    data_loader = DataLoader(
        data,
        batch_size=128,
        shuffle=False,
        pin_memory=(device.type == "cuda")
    )

    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for x, y in data_loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            output = model(x)

            loss = criterion(output, y)

            batch_size = y.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples

In [ ]:
def classification_accuracy(model, data):

    data_loader = DataLoader(
        data,
        batch_size=128,
        shuffle=False,
        pin_memory=(device.type == "cuda")
    )

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in data_loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            output = model(x)

            prediction = torch.argmax(
                output,
                dim=1
            )

            correct += (
                prediction == y
            ).sum().item()

            total += y.size(0)

    return 100 * correct / total

In [ ]:
def run_optimizer(optimizer_name):

    architecture_id = list(ARCHITECTURES.keys())[0]

    # =========================================================
    # DIRECTORIES
    # =========================================================

    os.makedirs("models", exist_ok=True)

    json_filename = (
        f"model_{architecture_id}_{optimizer_name}.json"
    )

    # =========================================================
    # LOAD EXISTING JSON IF IT EXISTS
    # This allows the notebook to resume after interruption.
    # =========================================================

    if os.path.exists(json_filename):

        print(f"Existing result file found: {json_filename}")
        print("Completed configurations will be skipped.")

        with open(json_filename, "r") as f:
            results = json.load(f)

    else:

        results = {
            "architecture": architecture_id,
            "activation": "tanh",
            "optimizer": optimizer_name,
            "learning_rate": LEARNING_RATE,
            "momentum": MOMENTUM,
            "threshold": THRESHOLD,
            "epochs_limit": epochs,
            "batch_size": optimizer_config[optimizer_name]["batch_size"],
            "configurations": {}
        }

    # =========================================================
    # START
    # =========================================================

    print("=" * 90)
    print(
        f"OPTIMIZER: {optimizer_name.upper()} | "
        f"ARCHITECTURE: {architecture_id} | "
        f"ACTIVATION: TANH"
    )
    print("=" * 90)

    # =========================================================
    # RUN ALL 3 CONFIGURATIONS
    # =========================================================

    for config_name, hidden_nodes in ARCHITECTURES[architecture_id].items():

        # -----------------------------------------------------
        # RESUME CHECK
        # -----------------------------------------------------

        if config_name in results["configurations"]:

            print("\n" + "-" * 90)
            print(
                f"SKIPPING {config_name} "
                f"(already completed)"
            )
            print("-" * 90)

            existing_result = results["configurations"][config_name]

            print(
                f"Saved Epochs: "
                f"{existing_result['epochs']}"
            )

            print(
                f"Saved Validation Accuracy: "
                f"{existing_result['final_val_accuracy']:.2f}%"
            )

            continue

        # -----------------------------------------------------
        # CONFIGURATION INFORMATION
        # -----------------------------------------------------

        print("\n" + "-" * 90)

        print(
            f"Architecture: {architecture_id} | "
            f"Configuration: {config_name} | "
            f"Neurons: {hidden_nodes} | "
            f"Optimizer: {optimizer_name}"
        )

        print("-" * 90)

        # =====================================================
        # BUILD MODEL
        # =====================================================

        model = build_model(hidden_nodes)

        # =====================================================
        # LOAD SAME INITIAL WEIGHTS
        # =====================================================

        initial_model_path = (
            f"initial/model_{architecture_id}_{config_name}.pth"
        )

        if not os.path.exists(initial_model_path):

            raise FileNotFoundError(
                f"Initial model not found:\n"
                f"{initial_model_path}"
            )

        model.load_state_dict(
            torch.load(
                initial_model_path,
                map_location=device
            )
        )

        model = model.to(device)

        # =====================================================
        # INITIAL ERROR
        # =====================================================

        initial_error = calculate_average_error(
            model,
            train_data
        )

        print(
            f"Initial Error: "
            f"{initial_error:.8f}"
        )

        # =====================================================
        # CREATE OPTIMIZER
        # =====================================================

        optimizer = optimizers[optimizer_name](model)

        # =====================================================
        # DATALOADER
        # =====================================================

        generator = torch.Generator()
        generator.manual_seed(SEED)

        train_loader = DataLoader(
            train_data,
            batch_size=optimizer_config[
                optimizer_name
            ]["batch_size"],
            shuffle=True,
            generator=generator,
            pin_memory=(device.type == "cuda")
        )

        # =====================================================
        # ERROR HISTORY
        #
        # errors[0] = error before training
        # errors[1] = error after epoch 1
        # etc.
        # =====================================================

        errors = [initial_error]

        # =====================================================
        # TRAINING
        # =====================================================

        for epoch in range(epochs):

            model.train()

            training_loss = 0.0
            total_samples = 0

            for x, y in train_loader:

                x = x.to(
                    device,
                    non_blocking=True
                )

                y = y.to(
                    device,
                    non_blocking=True
                )

                optimizer.zero_grad()

                output = model(x)

                loss = criterion(
                    output,
                    y
                )

                loss.backward()

                optimizer.step()

                current_batch_size = y.size(0)

                training_loss += (
                    loss.item() *
                    current_batch_size
                )

                total_samples += current_batch_size

            # -------------------------------------------------
            # AVERAGE TRAINING ERROR
            # -------------------------------------------------

            average_error = (
                training_loss /
                total_samples
            )

            errors.append(average_error)

            # -------------------------------------------------
            # SUCCESSIVE EPOCH ERROR DIFFERENCE
            # -------------------------------------------------

            error_difference = abs(
                errors[-1] -
                errors[-2]
            )

            # -------------------------------------------------
            # PROGRESS
            # -------------------------------------------------

            if (
                epoch == 0
                or (epoch + 1) % 10 == 0
            ):

                print(
                    f"Epoch {epoch + 1}: "
                    f"Error = {average_error:.8f} | "
                    f"Difference = "
                    f"{error_difference:.8f}"
                )

            # -------------------------------------------------
            # CONVERGENCE
            #
            # Assignment:
            # |E(t) - E(t-1)| < 1e-4
            # -------------------------------------------------

            if epoch>=2 and error_difference < THRESHOLD:

                print(
                    f"Converged at epoch "
                    f"{epoch + 1}"
                )

                break

        # =====================================================
        # FINAL METRICS
        # =====================================================

        final_error = errors[-1]

        converged_epochs = len(errors) - 1

        final_train_accuracy = (
            classification_accuracy(
                model,
                train_data
            )
        )

        final_val_accuracy = (
            classification_accuracy(
                model,
                val_data
            )
        )

        # =====================================================
        # SAVE TRAINED MODEL
        # =====================================================

        model_path = (
            f"models/"
            f"model_{architecture_id}_"
            f"{optimizer_name}_"
            f"{config_name}.pth"
        )

        torch.save(
            model.state_dict(),
            model_path
        )

        print(
            f"Saved trained model:\n"
            f"{model_path}"
        )

        # =====================================================
        # SAVE CONFIGURATION RESULT
        # =====================================================

        results["configurations"][config_name] = {

            "hidden_nodes": hidden_nodes,

            "initial_error": initial_error,

            "errors": errors,

            "epochs": converged_epochs,

            "final_error": final_error,

            "final_train_accuracy":
                final_train_accuracy,

            "final_val_accuracy":
                final_val_accuracy,

            "model_path": model_path
        }

        # =====================================================
        # SAVE JSON IMMEDIATELY
        #
        # IMPORTANT:
        # If Kaggle disconnects here, previous configurations
        # are already safely stored.
        # =====================================================

        with open(
            json_filename,
            "w"
        ) as f:

            json.dump(
                results,
                f,
                indent=4
            )

        print(
            f"Saved progress to:\n"
            f"{json_filename}"
        )

        print(
            f"Final Error: "
            f"{final_error:.8f}"
        )

        print(
            f"Final Train Accuracy: "
            f"{final_train_accuracy:.2f}%"
        )

        print(
            f"Final Validation Accuracy: "
            f"{final_val_accuracy:.2f}%"
        )

        print(
            f"Converged Epochs: "
            f"{converged_epochs}"
        )

        # =====================================================
        # FREE MEMORY BEFORE NEXT CONFIGURATION
        # =====================================================

        del model
        del optimizer
        del train_loader

        if device.type == "cuda":
            torch.cuda.empty_cache()

    # =========================================================
    # FINAL SUMMARY
    # =========================================================

    print("\n" + "=" * 90)

    print(
        f"COMPLETED OPTIMIZER: "
        f"{optimizer_name.upper()}"
    )

    print(
        f"Result file: "
        f"{json_filename}"
    )

    print("=" * 90)

    return results

In [ ]:
run_optimizer("sgd")

OPTIMIZER: SGD | ARCHITECTURE: 3 | ACTIVATION: TANH

------------------------------------------------------------------------------------------
Architecture: 3 | Configuration: config_1 | Neurons: [128, 64, 32] | Optimizer: sgd
------------------------------------------------------------------------------------------
Initial Error: 1.62585799
Epoch 1: Error = 0.74153598 | Difference = 0.88432200
Epoch 10: Error = 0.07591409 | Difference = 0.00743255
Epoch 20: Error = 0.03638570 | Difference = 0.00287765
Epoch 30: Error = 0.01796216 | Difference = 0.00126901
Epoch 40: Error = 0.00825664 | Difference = 0.00088492
Epoch 50: Error = 0.00365617 | Difference = 0.00042063
Converged at epoch 53
Saved trained model:
/kaggle/working/models/model_3_sgd_config_1.pth
Saved progress to:
model_3_sgd.json
Final Error: 0.00309348
Final Train Accuracy: 99.99%
Final Validation Accuracy: 97.71%
Converged Epochs: 53

------------------------------------------------------------------------------------------

{'architecture': 3,
 'activation': 'tanh',
 'optimizer': 'sgd',
 'learning_rate': 0.001,
 'momentum': 0.9,
 'threshold': 0.0001,
 'epochs_limit': 10000,
 'batch_size': 1,
 'configurations': {'config_1': {'hidden_nodes': [128, 64, 32],
   'initial_error': 1.6258579883650828,
   'errors': [1.6258579883650828,
    0.7415359845735888,
    0.18871964756129445,
    0.1483824734074801,
    0.12943537365286262,
    0.11672662482852567,
    0.10581328145110326,
    0.0979595940224993,
    0.09089881292169087,
    0.083346638078321,
    0.07591408676598899,
    0.070068085797589,
    0.06538575290212413,
    0.0606743340707713,
    0.05588394144292121,
    0.05183836391809328,
    0.04876986223120213,
    0.04491720154131731,
    0.04194596394813589,
    0.03926334863037656,
    0.03638570321967856,
    0.03421600707630364,
    0.03145957573787227,
    0.029005318525996394,
    0.027043836148768253,
    0.025175745829630804,
    0.023218882745603502,
    0.02171854146184982,
    0.02073345277716

In [ ]:
run_optimizer("sgd_general_delta")

OPTIMIZER: SGD_GENERAL_DELTA | ARCHITECTURE: 3 | ACTIVATION: TANH

------------------------------------------------------------------------------------------
Architecture: 3 | Configuration: config_1 | Neurons: [128, 64, 32] | Optimizer: sgd_general_delta
------------------------------------------------------------------------------------------
Initial Error: 1.62585799
Epoch 1: Error = 0.24009453 | Difference = 1.38576346
Epoch 10: Error = 0.01854357 | Difference = 0.00323337
Epoch 20: Error = 0.00067309 | Difference = 0.00058362
Converged at epoch 22
Saved trained model:
/kaggle/working/models/model_3_sgd_general_delta_config_1.pth
Saved progress to:
model_3_sgd_general_delta.json
Final Error: 0.00042080
Final Train Accuracy: 100.00%
Final Validation Accuracy: 98.37%
Converged Epochs: 22

------------------------------------------------------------------------------------------
Architecture: 3 | Configuration: config_2 | Neurons: [256, 128, 64] | Optimizer: sgd_general_delta
--------

{'architecture': 3,
 'activation': 'tanh',
 'optimizer': 'sgd_general_delta',
 'learning_rate': 0.001,
 'momentum': 0.9,
 'threshold': 0.0001,
 'epochs_limit': 10000,
 'batch_size': 1,
 'configurations': {'config_1': {'hidden_nodes': [128, 64, 32],
   'initial_error': 1.6258579883650828,
   'errors': [1.6258579883650828,
    0.24009453216381024,
    0.11633937428672161,
    0.08480479331754145,
    0.06440106651127143,
    0.050006800893598866,
    0.04185806635949002,
    0.03265373842008051,
    0.028492192850908888,
    0.021776945904668513,
    0.018543573722667957,
    0.013798897317153976,
    0.012832084078065051,
    0.008782646573403084,
    0.011386667109607113,
    0.0077723272632668474,
    0.004368158250756418,
    0.004077257436930565,
    0.0016049449982288625,
    0.0012567148430506624,
    0.0006730903042549039,
    0.0004842870038305501,
    0.00042080056968274405],
   'epochs': 22,
   'final_error': 0.00042080056968274405,
   'final_train_accuracy': 100.0,
   'final_

In [ ]:
run_optimizer("sgd_nag")

In [ ]:
run_optimizer("adam")

In [ ]:
run_optimizer("vanillaGD")

Existing result file found: model_3_vanillaGD.json
Completed configurations will be skipped.
OPTIMIZER: VANILLAGD | ARCHITECTURE: 3 | ACTIVATION: TANH

------------------------------------------------------------------------------------------
SKIPPING config_1 (already completed)
------------------------------------------------------------------------------------------
Saved Epochs: 3
Saved Validation Accuracy: 20.00%

------------------------------------------------------------------------------------------
SKIPPING config_2 (already completed)
------------------------------------------------------------------------------------------
Saved Epochs: 3
Saved Validation Accuracy: 19.68%

------------------------------------------------------------------------------------------
Architecture: 3 | Configuration: config_3 | Neurons: [512, 256, 128] | Optimizer: vanillaGD
------------------------------------------------------------------------------------------
Initial Error: 1.61168864
Epoch 

In [ ]:
run_optimizer("adagrad")

Existing result file found: model_3_adagrad.json
Completed configurations will be skipped.
OPTIMIZER: ADAGRAD | ARCHITECTURE: 3 | ACTIVATION: TANH

------------------------------------------------------------------------------------------
SKIPPING config_1 (already completed)
------------------------------------------------------------------------------------------
Saved Epochs: 1
Saved Validation Accuracy: 21.37%

------------------------------------------------------------------------------------------
SKIPPING config_2 (already completed)
------------------------------------------------------------------------------------------
Saved Epochs: 1
Saved Validation Accuracy: 46.64%

------------------------------------------------------------------------------------------
SKIPPING config_3 (already completed)
------------------------------------------------------------------------------------------
Saved Epochs: 1
Saved Validation Accuracy: 85.40%

COMPLETED OPTIMIZER: ADAGRAD
Result fil

{'architecture': 3,
 'activation': 'tanh',
 'optimizer': 'adagrad',
 'learning_rate': 0.001,
 'momentum': 0.9,
 'threshold': 0.0001,
 'epochs_limit': 10000,
 'batch_size': 11385,
 'configurations': {'config_1': {'hidden_nodes': [128, 64, 32],
   'initial_error': 1.6258579883650828,
   'errors': [1.6258579883650828, 1.625857949256897],
   'epochs': 1,
   'final_error': 1.625857949256897,
   'final_train_accuracy': 21.24725516029864,
   'final_val_accuracy': 21.370223978919633,
   'model_path': '/kaggle/working/models/model_3_adagrad_config_1.pth'},
  'config_2': {'hidden_nodes': [256, 128, 64],
   'initial_error': 1.6126577221305423,
   'errors': [1.6126577221305423, 1.6126577854156494],
   'epochs': 1,
   'final_error': 1.6126577854156494,
   'final_train_accuracy': 45.27009222661397,
   'final_val_accuracy': 46.640316205533594,
   'model_path': '/kaggle/working/models/model_3_adagrad_config_2.pth'},
  'config_3': {'hidden_nodes': [512, 256, 128],
   'initial_error': 1.611688635681196,

In [ ]:
run_optimizer("rmsprop")

In [ ]:
import matplotlib.pyplot as plt
import os
import json


# ============================================================
# LOAD ALL OPTIMIZER RESULTS
# ============================================================

def load_optimizer_results():

    architecture_id = list(ARCHITECTURES.keys())[0]

    all_results = {}

    for optimizer_name in optimizer_config:

        json_filename = (
            f"model_{architecture_id}_"
            f"{optimizer_name}.json"
        )

        if os.path.exists(json_filename):

            with open(
                json_filename,
                "r"
            ) as f:

                all_results[optimizer_name] = json.load(f)

        else:

            print(
                f"WARNING: "
                f"{json_filename} not found"
            )

    return all_results


# ============================================================
# PLOT TRAINING ERROR FOR EACH CONFIGURATION
# ============================================================

def plot_training_errors():

    architecture_id = list(ARCHITECTURES.keys())[0]

    all_results = load_optimizer_results()

    for config_name, hidden_nodes in ARCHITECTURES[
        architecture_id
    ].items():

        plt.figure(figsize=(10, 6))

        for optimizer_name in optimizer_config:

            if optimizer_name not in all_results:
                continue

            optimizer_results = all_results[
                optimizer_name
            ]

            if config_name not in optimizer_results[
                "configurations"
            ]:
                continue

            errors = optimizer_results[
                "configurations"
            ][
                config_name
            ][
                "errors"
            ]

            epochs_axis = range(
                len(errors)
            )

            plt.plot(
                epochs_axis,
                errors,
                label=optimizer_name
            )

        plt.xlabel("Epochs")

        plt.ylabel(
            "Average Training Error"
        )

        plt.title(
            f"Architecture {architecture_id} | "
            f"{config_name} | "
            f"Neurons: {hidden_nodes} | "
            f"Tanh"
        )

        plt.legend()

        plt.grid(True)

        plt.tight_layout()

        plt.show()


# ============================================================
# RUN PLOTS
# ============================================================

plot_training_errors()